In [1]:
from app.models import Embedding, SubTopic, Bookmark, Document_Entity_Link, Entity, Document
from app.db_connector import get_engine
import app.getters as getter
import app.icog_util as util
from app.prompt_models import ResponseWithIndex, DocumentPromptVerbatim
from app.transformers_util import generate_embeddings, get_util
from app.together_api_client import TogetherMixtralClient
from sqlalchemy.orm import Session
from sqlalchemy import (
    text,
    and_, or_, select
)

engine = get_engine()
user_id = 'HqAXhad3jrUWmPibnMf1xZczNIq2'


2024-06-19 09:48:41 - Connecting to database using TCP
2024-06-19 09:48:41 - Connected to database using TCP


[nltk_data] Downloading package punkt to /home/eboraks/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


2024-06-19 09:48:45 - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2024-06-19 09:48:45 - Starting new HTTPS connection (1): huggingface.co:443
2024-06-19 09:48:45 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json HTTP/1.1" 200 0
2024-06-19 09:48:45 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json HTTP/1.1" 200 0
2024-06-19 09:48:45 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/README.md HTTP/1.1" 200 0
2024-06-19 09:48:45 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json HTTP/1.1" 200 0
2024-06-19 09:48:45 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/sentence_bert_config.json HTTP/1.1" 200 0
2024-06-19 09:48:45 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/co

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [2]:
docs = getter.get_documents_by_user_id(user_id)

In [9]:
d1 = docs[4]
sentences = util.original_text_to_sentences(d1.original_text)
new_text = util.sentences_to_text(sentences)

In [10]:
d1.original_text

'Taylor Alison Swift (born December 13, 1989) is an American singer-songwriter. A subject of widespread public interest with a vast fanbase, she has influenced the music industry, popular culture, and politics through her songwriting, artistry, entrepreneurship, and advocacy.\n\nSwift began professional songwriting at 14. She signed with Big Machine Records in 2005 and achieved prominence as a country pop singer with the albums Taylor Swift (2006) and Fearless (2008). Their singles "Teardrops on My Guitar", "Love Story", and "You Belong with Me" were crossover successes on country and pop radio formats and brought Swift mainstream fame. She experimented with rock on Speak Now (2010) and electronic styles on Red (2012); the latter featured her first Billboard Hot 100 number-one single, "We Are Never Ever Getting Back Together". Swift recalibrated her image from country to pop with 1989 (2014), a synth-pop album containing the chart-topping songs "Shake It Off", "Blank Space", and "Bad B

In [11]:
new_text

'[0] Taylor Alison Swift (born December 13, 1989) is an American singer-songwriter. [1] A subject of widespread public interest with a vast fanbase, she has influenced the music industry, popular culture, and politics through her songwriting, artistry, entrepreneurship, and advocacy. [2] Swift began professional songwriting at 14. [3] She signed with Big Machine Records in 2005 and achieved prominence as a country pop singer with the albums Taylor Swift (2006) and Fearless (2008). [4] Their singles "Teardrops on My Guitar", "Love Story", and "You Belong with Me" were crossover successes on country and pop radio formats and brought Swift mainstream fame. [5] She experimented with rock on Speak Now (2010) and electronic styles on Red (2012); the latter featured her first Billboard Hot 100 number-one single, "We Are Never Ever Getting Back Together". [6] Swift recalibrated her image from country to pop with 1989 (2014), a synth-pop album containing the chart-topping songs "Shake It Off", 

In [12]:
client = TogetherMixtralClient()

2024-06-19 09:54:44 - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7f4d84e5bad0>


In [13]:
res = await client.generate(DocumentPromptVerbatim, DocumentPromptVerbatim.get_messages(new_text))

2024-06-19 09:54:47 - Attempt 1 to generate summary
2024-06-19 09:56:22 - Response status: finished
2024-06-19 09:56:22 - Answer is not None. Stop retrying. Number of attempts 1


In [14]:
print(res.whatThisArticleIsAbout.answer)
print(res.whatThisArticleIsAbout.sentences_indcies) 
for ind in res.whatThisArticleIsAbout.sentences_indcies:
    print(sentences[ind])

This article is about Taylor Alison Swift, an American singer-songwriter who has significantly influenced the music industry, popular culture, and politics through her songwriting, artistry, entrepreneurship, and advocacy. Swift began professional songwriting at 14 and achieved prominence as a country pop singer with the albums Taylor Swift (2006) and Fearless (2008). She experimented with rock on Speak Now (2010) and electronic styles on Red (2012). Swift recalibrated her image from country to pop with 1989 (2014), a synth-pop album containing the chart-topping songs 'Shake It Off', 'Blank Space', and 'Bad Blood'. Media scrutiny inspired the hip-hop-influenced Reputation (2017) and its number-one single 'Look What You Made Me Do'. After signing with Republic Records in 2018, Swift released the eclectic pop album Lover (2019) and the autobiographical documentary Miss Americana (2020). She explored indie folk styles on the 2020 albums Folklore and Evermore, subdued pop genres on Midnigh